# Capítulo 5 · Actividad: ¿quién sobrevivió al Titanic?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/01_actividad_titanic_arbol.ipynb)

Entrena un **árbol de decisión** que prediga si un pasajero sobrevivió (`survived`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

## 🗺️ El orden de los pasos

```
1. Mirar            → conocer los datos
2. X / y            → qué ve el modelo y qué adivina; quitar fugas
3. Split            → apartar el examen ANTES de aprender nada
4. Preprocesar      → fit SOLO con train
5. Modelo base      → punto de partida
6. Validación cruzada → medir sin tocar test
7. Optimizar        → elegir hiperparámetros con CV
8. Test             → examen final, una sola vez
9. Entender         → árbol e importancias
10. Conclusiones
```

**Regla que explica todo el orden:** lo que *aprende* de los datos va después del split y se ajusta solo con train.

---
## Paso 1 · Mirar los datos

Antes de decidir nada, **conoce lo que tienes**: filas, columnas, tipos, faltantes y el % de supervivientes.

**¿Por qué va primero?** No puedes elegir qué limpiar sin saber qué está sucio.

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df["survived"].value_counts(normalize=True) * 100

📖 **Qué hace y por qué**

- `info()`: tipos y no nulos. `age` (714/891), `embarked` (889) y `deck` (203) tienen huecos.
- `isna().sum()`: los mismos huecos, contados.
- `value_counts(normalize=True) * 100`: **el 61.6 % no sobrevivió**. Un modelo que diga siempre "no" acierta el 61.6 %: esa es la **línea base** que hay que superar.

✍️ **Respuesta / interpretación:**

El 61.6 % no sobrevivió, así que un modelo que diga siempre "no sobrevive" acierta el 61.6 %. Esa es mi línea base. Lo que más me preocupa es deck, con 688 de 891 vacíos (77 %), y luego age, con 177 vacíos (20 %).

---
## Paso 2 · Separar X e y y quitar columnas peligrosas

- `y` = lo que el modelo **adivina** (`survived`).
- `X` = lo que el modelo **ve**. De aquí salen las columnas que chivan la respuesta (fuga) y las redundantes.

**¿Por qué va antes del split?** Quitar una columna no *aprende* nada de los datos, así que no contamina el test.

In [ ]:
# y = lo que el modelo tiene que adivinar
y = df["survived"]

# X = lo que el modelo puede ver (quito la respuesta, la fuga y las repetidas)
X = df.drop(columns=["survived", "alive", "class", "embark_town", "who", "adult_male", "alone"])

# deck: falta casi todo, así que solo guardo si se conoce (1) o no (0)
X["tiene_deck"] = X["deck"].notna().astype(int)
X = X.drop(columns=["deck"])

print(X.shape)
print(y.shape)
print(X.columns)

📖 **Qué hace y por qué**

- `X` es una **tabla** (8 columnas) e `y` una **columna**. El modelo aprende la relación X → y.
- `tiene_deck` puede ir **antes** del split porque cada fila solo se mira a sí misma: no calcula nada con los demás pasajeros.
- Rellenar `age` con la mediana, en cambio, **sí** mira a todos → tiene que esperar al paso 4.

✍️ **Tu interpretación:** ¿Qué pasaría si dejaras `alive` en X? ¿Ese modelo serviría con un pasajero nuevo?

---
## Paso 3 · Separar train y test

80 % para estudiar, 20 % para el examen final. Estratificado y con `random_state=42`.

**¿Por qué va aquí, antes de imputar o codificar?** Todo lo que *aprenda* de los datos (medianas, modas, categorías) tiene que salir **solo de train**. Si no, hay fuga de información.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("% sobrevivientes en train:", y_train.mean() * 100)
print("% sobrevivientes en test:", y_test.mean() * 100)

📖 **Qué hace y por qué**

- `train_test_split` devuelve **4 cosas en este orden**: `X_train`, `X_test`, `y_train`, `y_test`.
- `test_size=0.2`: el 20 % va a test (179 pasajeros) y el 80 % a train (712).
- `stratify=y`: los dos grupos quedan con el mismo % de supervivientes (≈38 %).
- `random_state=42`: el reparto sale igual cada vez que ejecutas.
- 🔒 **Desde aquí, `X_test` e `y_test` no se usan para aprender nada.**

✍️ **Tu interpretación:** ¿Por qué los % de train y test salen casi iguales? ¿Qué pasaría sin `stratify`?

---
## Paso 4 · Preprocesamiento

Rellenar los huecos y convertir el texto en números. **Siempre en este orden: calculo con train → aplico a train → aplico a test.**

In [ ]:
# 1) age: rellenar los huecos con la mediana de TRAIN
mediana_age = X_train["age"].median()
print("Mediana de age (train):", mediana_age)

X_train["age"] = X_train["age"].fillna(mediana_age)
X_test["age"] = X_test["age"].fillna(mediana_age)

# 2) embarked: rellenar los huecos con la moda (el valor más común) de TRAIN
moda_embarked = X_train["embarked"].mode()[0]
print("Moda de embarked (train):", moda_embarked)

X_train["embarked"] = X_train["embarked"].fillna(moda_embarked)
X_test["embarked"] = X_test["embarked"].fillna(moda_embarked)

# 3) sex: texto -> número
X_train["sex"] = X_train["sex"].map({"male": 0, "female": 1})
X_test["sex"] = X_test["sex"].map({"male": 0, "female": 1})

# 4) embarked: una columna 0/1 por cada puerto (one-hot)
X_train = pd.get_dummies(X_train, columns=["embarked"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["embarked"], dtype=int)

# Comprobaciones
print("Huecos en train:", X_train.isna().sum().sum())
print("Huecos en test:", X_test.isna().sum().sum())
print("¿Mismas columnas en train y test?", list(X_train.columns) == list(X_test.columns))
X_train.head()

📖 **Qué hace y por qué**

- `median()` calcula la mediana **solo con `X_train`**. Luego `fillna(mediana_age)` rellena los huecos de train **y también los de test con ese mismo número**. Si la calculara con todo `X`, el test estaría "ayudando" a preparar los datos: eso es fuga de información.
- `mode()[0]`: `mode()` devuelve una lista de modas; `[0]` toma la primera.
- `map({"male": 0, "female": 1})` cambia cada texto por su número.
- `pd.get_dummies` convierte `embarked` en tres columnas: `embarked_C`, `embarked_Q` y `embarked_S`, con un 1 en la del puerto del pasajero.
- La última comprobación importa: train y test tienen que tener **las mismas columnas en el mismo orden**, o el modelo no funciona.
- **No escalo**: un árbol solo pregunta cosas como "¿age ≤ 9.5?". Cambiar la escala no cambia esas preguntas.

✍️ **Tu interpretación:** ¿Por qué la mediana de `age` se calcula solo con train y no con todo el dataset?

---
## Paso 5 · Modelo base

Un `DecisionTreeClassifier` con los valores por defecto (sin límites). Es el punto de partida para comparar.

In [ ]:
arbol_base = DecisionTreeClassifier(random_state=42)
arbol_base.fit(X_train, y_train)

pred_train = arbol_base.predict(X_train)
print("Accuracy en train:", accuracy_score(y_train, pred_train))
print("Profundidad:", arbol_base.get_depth())
print("Hojas:", arbol_base.get_n_leaves())

📖 **Qué hace y por qué**

- `DecisionTreeClassifier(random_state=42)`: creo el árbol (todavía no sabe nada).
- `fit(X_train, y_train)`: **entrena**. Busca las preguntas que mejor separan a los que sobrevivieron.
- `predict(X_train)`: le pido que adivine sobre los mismos datos con los que estudió.
- `accuracy_score`: el % de aciertos.
- Sale **≈98.6 % en train**, con 16 niveles y 164 hojas. Parece genial, pero el árbol se aprendió de memoria a los pasajeros de train. Eso no dice nada de pasajeros nuevos.

✍️ **Tu interpretación:** ¿Por qué un 98 % en train no prueba que el modelo sea bueno?

---
## Paso 6 · Validación cruzada

Medir el modelo base con `cross_val_score` **sobre train**, sin tocar test.

In [ ]:
scores = cross_val_score(arbol_base, X_train, y_train, cv=5)

print("Accuracy en cada parte:", scores)
print("Promedio:", scores.mean())

📖 **Qué hace y por qué**

- `cv=5`: parte train en 5 trozos. Entrena con 4 y examina con el que sobra. Lo repite 5 veces, cambiando el trozo de examen.
- Así obtengo 5 notas de exámenes de práctica **sin usar test**.
- Promedio **≈75.9 %** frente a **98.6 % en train**. La diferencia es enorme: el árbol **memorizó** (overfitting) en vez de aprender reglas generales.

✍️ **Tu interpretación:** Compara train (98.6 %) con la validación cruzada (≈75.9 %). ¿Qué le pasa al árbol y cómo lo frenarías?

---
## Paso 7 · Optimizar hiperparámetros

`GridSearchCV` prueba varias combinaciones de `max_depth` y `min_samples_leaf` con validación cruzada y se queda con la mejor.

In [ ]:
parametros = {
    "max_depth": [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 2, 4, 8],
}

busqueda = GridSearchCV(DecisionTreeClassifier(random_state=42), parametros, cv=5)
busqueda.fit(X_train, y_train)

print("Mejores parámetros:", busqueda.best_params_)
print("Accuracy de validación del mejor:", busqueda.best_score_)

arbol_opt = busqueda.best_estimator_

pred_train_opt = arbol_opt.predict(X_train)
print("Accuracy en train:", accuracy_score(y_train, pred_train_opt))
print("Profundidad:", arbol_opt.get_depth())
print("Hojas:", arbol_opt.get_n_leaves())

📖 **Qué hace y por qué**

- **Hiperparámetros** = ajustes que elijo yo antes de entrenar.
  - `max_depth`: cuántas preguntas seguidas puede hacer el árbol.
  - `min_samples_leaf`: el mínimo de pasajeros que debe tener cada hoja.
- `GridSearchCV` prueba las 6 × 4 = **24 combinaciones**, cada una con validación cruzada de 5 partes, y se queda con la de mejor promedio. **No usa test.**
- `best_estimator_` es el árbol ganador, ya entrenado con todo train.
- Gana `max_depth=5` y `min_samples_leaf=2`: **23 hojas en lugar de 164**. Baja en train (≈85.7 %) pero **sube en validación (≈81.9 %)**. Train y validación quedan cerca: ya no memoriza.

✍️ **Tu interpretación:** ¿Por qué el árbol más pequeño valida mejor que el grande?

---
## Paso 8 · Evaluar en test (una sola vez)

Accuracy, matriz de confusión y `classification_report` del modelo base y del optimizado.

**¿Por qué al final y una sola vez?** El test simula pasajeros nuevos. Solo vale si el modelo nunca lo vio antes.

In [ ]:
# Modelo base en test
pred_test_base = arbol_base.predict(X_test)
print("Accuracy en test (base):", accuracy_score(y_test, pred_test_base))

# Modelo optimizado en test
pred_test_opt = arbol_opt.predict(X_test)
print("Accuracy en test (optimizado):", accuracy_score(y_test, pred_test_opt))

print()
print("Matriz de confusión (optimizado):")
print(confusion_matrix(y_test, pred_test_opt))

print()
print(classification_report(y_test, pred_test_opt))

📖 **Qué hace y por qué**

- Es el **examen final**: la primera y única vez que los modelos ven `X_test`.
- **Accuracy en test**: base ≈74.9 % y optimizado ≈76.5 %. Los dos superan la línea base (61.6 %), y el optimizado gana.
- **Matriz de confusión**: las filas son la realidad (0 = murió, 1 = sobrevivió) y las columnas lo que predijo el modelo:

  ```
                  predijo 0   predijo 1
  real 0 (murió)     91          19
  real 1 (vivió)     23          46
  ```
  La diagonal (91 y 46) son los aciertos.
- `classification_report`, en la fila `1` (sobrevivió):
  - **precision ≈ 0.71**: de los que el modelo dijo "sobrevive", el 71 % sobrevivió de verdad.
  - **recall ≈ 0.67**: de los que sobrevivieron, el modelo encontró al 67 %.

✍️ **Tu interpretación:** Mirando la matriz de confusión, ¿el modelo se equivoca más con los que murieron o con los que sobrevivieron?

---
## Paso 9 · Entender el modelo

Dibuja el árbol (`plot_tree`) y mira `feature_importances_`. ¿Coincide con lo que encontraste en el EDA del capítulo 3?

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(arbol_opt, feature_names=X_train.columns, class_names=["No", "Sí"], filled=True)
plt.show()

importancias = pd.Series(arbol_opt.feature_importances_, index=X_train.columns)
print(importancias.sort_values(ascending=False))

📖 **Qué hace y por qué**

- `plot_tree` dibuja el árbol. Cada caja es una pregunta. Se lee de arriba abajo: **izquierda = sí se cumple**, derecha = no. El color indica qué clase domina.
- La primera pregunta es sobre `sex`: es la que mejor separa, como viste en el EDA del capítulo 3.
- `feature_importances_` dice cuánto usó el árbol cada columna (suman 1):
  - `sex` ≈ 0.53
  - `pclass` ≈ 0.13
  - `age` ≈ 0.13
  - `fare` ≈ 0.08
  - `tiene_deck` ≈ 0.07
- Los puertos casi no aportan: el puerto solo reflejaba la clase.

✍️ **Tu interpretación:** ¿Coincide la importancia de las variables con lo que encontraste en el EDA del capítulo 3?

---
## Paso 10 · Conclusiones

- ¿Cuánto mejoraste sobre la línea base y sobre el modelo base?
- ¿Qué variables usa más el árbol?
- ¿Qué decisión de preprocesamiento cambiarías?

✍️ **Tus conclusiones** (escríbelas con tus palabras, apoyándote en los números de arriba):

- ¿Cuánto mejoraste sobre la línea base (61.6 %) y sobre el modelo base?
- ¿Qué tres variables usa más el árbol? ¿Tiene sentido histórico?
- ¿Qué decisión de preprocesamiento cambiarías para intentar mejorar? (p. ej. tamaño de la familia, tramos de edad, añadir `min_samples_split` a la rejilla)